# Impor packages

In [1]:
# pip install pandas
# pip install numpy
# pip install datetime
import pandas as pd
import numpy as np
import datetime as dt

# Impor data dari CSV ke DataFrame

In [2]:
df = pd.read_csv('Online Retail Data.csv', header=0)
df

,order_id,product_code,product_name,quantity,order_date,price,customer_id
0,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346.0
1,C493411,21539,RETRO SPOTS BUTTER DISH,-1,2010-01-04 09:43:00,4.25,14590.0
2,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346.0
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1,2010-01-04 09:54:00,0.85,NaN
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1,2010-01-04 09:54:00,3.75,NaN
...,...,...,...,...,...,...,...
71295,501080,22243,"HOOK, 5 HANGER ,MAGIC TOADSTOOL RED",2,2010-03-12 11:19:00,3.36,NaN
71296,501080,22245,"HOOK, 1 HANGER ,MAGIC GARDEN",2,2010-03-12 11:19:00,1.66,NaN
71297,501080,22249,DECORATION WHITE CHICK MAGIC GARDEN,1,2010-03-12 11:19:00,1.66,NaN
71298,501080,22275,WEEKEND BAG VINTAGE ROSE PAISLEY,1,2010-03-12 11:19:00,16.98,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71300 entries, 0 to 71299
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      71300 non-null  object 
 1   product_code  71300 non-null  object 
 2   product_name  70397 non-null  object 
 3   quantity      71300 non-null  int64  
 4   order_date    71300 non-null  object 
 5   price         71300 non-null  float64
 6   customer_id   53248 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 3.8+ MB


# Data cleansing

In [9]:
df_clean = df.copy()
# membuat kolom date
df_clean['date'] = pd.to_datetime(df_clean['order_date']).dt.date
# menghapus semua baris tanpa customer_id
df_clean = df_clean[~df_clean['customer_id'].isna()]
# mengkonversi customer_id menjadi string
df_clean['customer_id'] = df_clean['customer_id'].astype(str)
# menghapus semua baris tanpa product_name
df_clean = df_clean[~df_clean['product_name'].isna()]
# membuat semua product_name berhuruf kecil
df_clean['product_name'] = df_clean['product_name'].str.lower()
# menghapus semua baris dengan product_code atau product_name test
df_clean = df_clean[(~df_clean['product_code'].str.lower().str.contains('test')) |
                    (~df_clean['product_name'].str.contains('test '))]
# menghapus baris dengan status cancelled, yaitu yang order_id-nya diawali 'C'
df_clean = df_clean[df_clean['order_id'].str[:1]!='C']
# mengubah nilai quantity yang negatif menjadi positif karena nilai negatif tersebut hanya menandakan order tersebut cancelled
df_clean['quantity'] = df_clean['quantity'].abs()
# menghapus baris dengan price bernilai negatif
df_clean = df_clean[df_clean['price']>0]
# membuat nilai amount, yaitu perkalian antara quantity dan price
df_clean['amount'] = df_clean['quantity'] * df_clean['price']
# mengganti product_name dari product_code yang memiliki beberapa product_name dengan salah satu product_name-nya yang paling sering muncul
most_freq_product_name = df_clean.groupby(['product_code','product_name'], as_index=False).agg(order_cnt=('order_id','nunique')).sort_values(['product_code','order_cnt'], ascending=[True,False])
most_freq_product_name['rank'] = most_freq_product_name.groupby('product_code')['order_cnt'].rank(method='first', ascending=False)
most_freq_product_name = most_freq_product_name[most_freq_product_name['rank']==1].drop(columns=['order_cnt','rank'])
df_clean = df_clean.merge(most_freq_product_name.rename(columns={'product_name':'most_freq_product_name'}), how='left', on='product_code')
df_clean['product_name'] = df_clean['most_freq_product_name']
df_clean = df_clean.drop(columns='most_freq_product_name')
# menghapus outlier
from scipy import stats
df_clean = df_clean[(np.abs(stats.zscore(df_clean[['quantity','amount']]))<3).all(axis=1)]
df_clean = df_clean.reset_index(drop=True)
df_clean

,order_id,product_code,product_name,quantity,order_date,price,customer_id,date,amount
0,493414,21844,retro spot mug,36,2010-01-04 10:28:00,2.55,14590.0,2010-01-04,91.8
1,493414,21533,retro spot large milk jug,12,2010-01-04 10:28:00,4.25,14590.0,2010-01-04,51.0
2,493414,37508,new england ceramic cake server,2,2010-01-04 10:28:00,2.55,14590.0,2010-01-04,5.1
3,493414,35001G,hand open shape gold,2,2010-01-04 10:28:00,4.25,14590.0,2010-01-04,8.5
4,493414,21527,retro spot traditional teapot,12,2010-01-04 10:28:00,6.95,14590.0,2010-01-04,83.4
...,...,...,...,...,...,...,...,...,...
51213,501079,21980,pack of 12 red spotty tissues,50,2010-03-12 11:13:00,0.40,15008.0,2010-03-12,20.0
51214,501079,21981,pack of 12 woodland tissues,50,2010-03-12 11:13:00,0.40,15008.0,2010-03-12,20.0
51215,501079,21986,pack of 12 pink spot tissues,50,2010-03-12 11:13:00,0.40,15008.0,2010-03-12,20.0
51216,501079,21983,pack of 12 blue paisley tissues,50,2010-03-12 11:13:00,0.40,15008.0,2010-03-12,20.0


In [10]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51218 entries, 0 to 51217
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      51218 non-null  object 
 1   product_code  51218 non-null  object 
 2   product_name  51218 non-null  object 
 3   quantity      51218 non-null  int64  
 4   order_date    51218 non-null  object 
 5   price         51218 non-null  float64
 6   customer_id   51218 non-null  object 
 7   date          51218 non-null  object 
 8   amount        51218 non-null  float64
dtypes: float64(2), int64(1), object(6)
memory usage: 3.5+ MB


# Menyiapkan data basket

## Buat DataFrame basket

In [11]:
basket = pd.pivot_table(df_clean, index='order_id', columns='product_name', values='product_code', aggfunc='nunique', fill_value=0)
basket

product_name,12 daisy pegs in wood box,12 egg house painted wood,12 ivory rose peg place settings,12 mini toadstool pegs,12 pencils small tube posy,12 pencils small tube red spotty,12 pencils small tube skull,12 pencils tall tube posy,12 pencils tall tube red spotty,12 pencils tall tube skulls,...,you're confusing me metal sign,zinc finish 15cm planter pots,zinc heart lattice 2 wall planter,zinc heart lattice double planter,zinc heart lattice planter bowl,zinc heart lattice t-light holder,zinc heart lattice tray oval,zinc metal heart decoration,zinc top 2 door wooden shelf,zinc willie winkie candle stick
order_id,,,,,,,,,,,,,,,,,,,,,
493414,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493427,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493428,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493432,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493433,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501039,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
501044,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
501045,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
basket.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2346 entries, 493414 to 501079
Columns: 2815 entries, 12 daisy pegs in wood box to zinc willie winkie  candle stick
dtypes: int64(2815)
memory usage: 50.4+ MB


## Encode DataFrame basket dengan nilai True untuk semua nilai di atas 0 dan False untuk semua nilai 0

In [13]:
def encode(x):
    if x==0:
        return False
    if x>0:
        return True

basket_encode = basket.applymap(encode)
basket_encode

<ipython-input-13-31958d44448d>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_encode = basket.applymap(encode)


product_name,12 daisy pegs in wood box,12 egg house painted wood,12 ivory rose peg place settings,12 mini toadstool pegs,12 pencils small tube posy,12 pencils small tube red spotty,12 pencils small tube skull,12 pencils tall tube posy,12 pencils tall tube red spotty,12 pencils tall tube skulls,...,you're confusing me metal sign,zinc finish 15cm planter pots,zinc heart lattice 2 wall planter,zinc heart lattice double planter,zinc heart lattice planter bowl,zinc heart lattice t-light holder,zinc heart lattice tray oval,zinc metal heart decoration,zinc top 2 door wooden shelf,zinc willie winkie candle stick
order_id,,,,,,,,,,,,,,,,,,,,,
493414,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493427,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493428,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493432,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493433,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501039,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
501044,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
501045,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [14]:
basket_encode.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2346 entries, 493414 to 501079
Columns: 2815 entries, 12 daisy pegs in wood box to zinc willie winkie  candle stick
dtypes: bool(2815)
memory usage: 6.3+ MB


## Ambil transaksi dengan banyaknya produk unik lebih dari 1 saja

In [15]:
basket_filter = basket_encode[(basket_encode>0).sum(axis=1)>1]
basket_filter

product_name,12 daisy pegs in wood box,12 egg house painted wood,12 ivory rose peg place settings,12 mini toadstool pegs,12 pencils small tube posy,12 pencils small tube red spotty,12 pencils small tube skull,12 pencils tall tube posy,12 pencils tall tube red spotty,12 pencils tall tube skulls,...,you're confusing me metal sign,zinc finish 15cm planter pots,zinc heart lattice 2 wall planter,zinc heart lattice double planter,zinc heart lattice planter bowl,zinc heart lattice t-light holder,zinc heart lattice tray oval,zinc metal heart decoration,zinc top 2 door wooden shelf,zinc willie winkie candle stick
order_id,,,,,,,,,,,,,,,,,,,,,
493414,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493427,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493428,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493432,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493433,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501039,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
501044,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
501045,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [16]:
basket_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2193 entries, 493414 to 501079
Columns: 2815 entries, 12 daisy pegs in wood box to zinc willie winkie  candle stick
dtypes: bool(2815)
memory usage: 5.9+ MB


# Mengaplikasikan apriori algorithm

## Buat list frequent itemset (kumpulan produk yang sering dibeli)

In [17]:
# pip install mlxtend
from mlxtend.frequent_patterns import apriori

frequent_itemset = apriori(basket_filter, min_support=.01, use_colnames=True).sort_values('support', ascending=False).reset_index(drop=True)
frequent_itemset['product_cnt'] = frequent_itemset['itemsets'].apply(lambda x: len(x))
frequent_itemset

,support,itemsets,product_cnt
0,0.222070,(white hanging heart t-light holder),1
1,0.104423,(door mat union flag),1
2,0.096215,(jumbo bag red white spotty),1
3,0.088007,(home building block word),1
4,0.086639,(wooden frame antique white),1
...,...,...,...
1727,0.010032,"(gin + tonic diet metal sign, cream heart card...",2
1728,0.010032,"(chocolate this way metal sign, hand over the ...",2
1729,0.010032,"(pack of 12 hearts design tissues, red hanging...",2
1730,0.010032,"(childs apron spaceboy design, lunch bag pink ...",2


## Hitung nilai support, confidence, dan lift dari setiap pasangan produk yang mungkin

In [18]:
from mlxtend.frequent_patterns import association_rules

product_association = association_rules(frequent_itemset, metric='confidence', min_threshold=.7).sort_values(['support','confidence'], ascending=[False,False]).reset_index(drop=True)
product_association

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(red hanging heart t-light holder),(white hanging heart t-light holder),0.086639,0.222070,0.069767,0.805263,3.626164,1.0,0.050527,3.994775,0.792925,0.291985,0.749673,0.559716
1,(blue felt easter egg basket),(pink felt easter egg basket),0.043320,0.051072,0.038304,0.884211,17.313158,1.0,0.036091,8.195291,0.984906,0.682927,0.877979,0.817105
2,(pink felt easter egg basket),(blue felt easter egg basket),0.051072,0.043320,0.038304,0.750000,17.313158,1.0,0.036091,3.826721,0.992952,0.682927,0.738680,0.817105
3,(cream felt easter egg basket),(pink felt easter egg basket),0.047880,0.051072,0.034200,0.714286,13.985969,1.0,0.031754,3.321249,0.975192,0.528169,0.698908,0.691964
4,(toy tidy spaceboy),(toy tidy pink retrospot),0.040128,0.050616,0.031464,0.784091,15.491093,1.0,0.029433,4.397149,0.974553,0.530769,0.772580,0.702856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
387,"(lunchbox with cutlery retrospot, red hanging ...",(wooden frame antique white),0.014136,0.086639,0.010032,0.709677,8.191171,1.0,0.008807,3.146020,0.890505,0.110553,0.682138,0.412733
388,"(lunchbox with cutlery retrospot, red hanging ...",(wooden frame antique white),0.014136,0.086639,0.010032,0.709677,8.191171,1.0,0.008807,3.146020,0.890505,0.110553,0.682138,0.412733
389,"(lunchbox with cutlery retrospot, red hanging ...","(wooden frame antique white, white hanging hea...",0.014136,0.042864,0.010032,0.709677,16.556623,1.0,0.009426,3.296803,0.953074,0.213592,0.696676,0.471860
390,"(wooden frame antique white, fancy font home s...",(wood s/3 cabinet ant white finish),0.014136,0.055632,0.010032,0.709677,12.756742,1.0,0.009246,3.252825,0.934825,0.167939,0.692575,0.445003
